# Microproyecto 2 — Clasificación de textos según los Objetivos de Desarrollo Sostenible

**Machine Learning No Supervisado — MAIA, Universidad de los Andes**

El objetivo es construir una solución de procesamiento de lenguaje natural que asigne a un
texto en español el ODS con el que guarda mayor relación semántica, para apoyar el análisis
de información textual en procesos de planeación participativa.

El notebook sigue el orden de la rúbrica de evaluación:

| Sección | Actividad | Peso |
|---|---|---|
| 2–3 | Preparación de los datos y reducción de dimensionalidad, con justificación | 30% |
| 4 | Construcción del pipeline de preparación | 15% |
| 5 | Modelo LSA sobre la matriz TF-IDF e interpretación de al menos 5 tópicos | 15% |
| 6 | Clasificador con búsqueda de hiperparámetros y métricas justificadas | 30% |
| 7 | Desempeño sobre textos no usados en el aprendizaje (al menos 4) | 10% |

> **Cómo usar este archivo.** Las celdas de código funcionan tal como están y sirven de
> andamiaje. Los bloques marcados con **✍️ Justificación** son los que califica la rúbrica:
> ahí va tu argumentación, no basta con el resultado numérico.

## 1. Configuración e importaciones

In [ ]:
import sys
import warnings
from pathlib import Path

# El notebook vive en notebooks/, pero los modulos y los datos cuelgan de la raiz
# del proyecto. Se agrega la raiz al path para poder importar `src`.
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

from src.DataPreprocessing import (
    NOMBRES_ODS,
    SEED,
    cargar_datos,
    construir_pipeline_preparacion,
    etiquetas_ods,
    terminos_por_topico,
)

warnings.filterwarnings("ignore")
np.random.seed(SEED)
pd.set_option("display.max_colwidth", 120)

print("Raiz del proyecto:", RAIZ)

## 2. Carga y exploración del conjunto de datos

Los textos provienen del *OSDG Community Dataset* (2023), traducidos al español y
aumentados con la API de ChatGPT. Cada registro es un texto etiquetado por voluntarios
con el ODS al que se refiere.

In [ ]:
df = cargar_datos(RAIZ / "data" / "Datos_textosODS.xlsx")
print(f"Registros: {len(df):,}  |  Columnas: {list(df.columns)}")
df.head(3)

In [ ]:
# Calidad basica: nulos y duplicados
print("Nulos por columna:")
print(df.isna().sum())
print(f"\nTextos duplicados: {df['textos'].duplicated().sum()}")

In [ ]:
# Distribucion de clases
conteo = df["ODS"].value_counts().sort_index()
resumen = pd.DataFrame({
    "ODS": conteo.index,
    "nombre": [NOMBRES_ODS[c] for c in conteo.index],
    "n_textos": conteo.values,
    "porcentaje": (conteo.values / len(df) * 100).round(2),
})
display(resumen)

fig, ax = plt.subplots(figsize=(11, 4))
sns.barplot(x=conteo.index, y=conteo.values, ax=ax, color="#4C78A8")
ax.set(xlabel="ODS", ylabel="Numero de textos", title="Distribucion de textos por ODS")
plt.tight_layout()
plt.show()

print(f"Clases presentes: {sorted(df['ODS'].unique())}")
print(f"Razon de desbalance (mayoritaria / minoritaria): {conteo.max() / conteo.min():.2f}")

### Dos hallazgos que condicionan el modelado

1. **Hay 16 clases, no 17.** El enunciado habla de los 17 ODS, pero el ODS 17
   (*Alianzas para lograr los objetivos*) no aparece en los datos. Las etiquetas de los
   reportes deben derivarse de los datos, no fijarse en 17.
2. **Las clases están desbalanceadas** (del orden de 3 a 1 entre la más y la menos
   frecuente). Esto tiene dos consecuencias: las particiones deben ser **estratificadas**
   y la métrica principal debe ser el **F1 macro**, que pondera igual a todas las clases;
   la exactitud premiaría a un modelo que ignore las clases minoritarias.

In [ ]:
# Longitud de los textos: sirve para decidir min_df / max_features mas adelante
df["n_caracteres"] = df["textos"].str.len()
df["n_palabras"] = df["textos"].str.split().str.len()
display(df[["n_caracteres", "n_palabras"]].describe().round(1))

## 3. Partición en entrenamiento y prueba

La partición se hace **antes** de ajustar cualquier transformación. Si el TF-IDF o el SVD
se ajustaran con todos los datos, el vocabulario y las componentes latentes incorporarían
información del conjunto de prueba y la evaluación quedaría contaminada.

In [ ]:
X = df["textos"]
y = df["ODS"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

print(f"Entrenamiento: {len(X_train):,}   Prueba: {len(X_test):,}")
print("\nProporcion de clases conservada por la estratificacion:")
display(pd.DataFrame({
    "train_%": (y_train.value_counts(normalize=True).sort_index() * 100).round(2),
    "test_%": (y_test.value_counts(normalize=True).sort_index() * 100).round(2),
}))

## 4. Pipeline de preparación de datos

El pipeline encadena tres pasos, definidos en [`src/DataPreprocessing.py`](../src/DataPreprocessing.py):

| Paso | Qué hace | Por qué |
|---|---|---|
| `limpieza` | minúsculas, eliminación de acentos, tokenización, stopwords en español, stemming con `SnowballStemmer` | los textos están en español: el `PorterStemmer` de los tutoriales es de inglés y no aplica |
| `tfidf` | bolsa de palabras con pesado TF-IDF | penaliza términos omnipresentes y resalta los discriminativos entre ODS |
| `lsa` | `TruncatedSVD` | reduce la dimensionalidad y **opera directamente sobre la matriz dispersa**, a diferencia de PCA, que exigiría densificarla |

Mantener la preparación dentro de un `Pipeline` no es solo estilo: garantiza que en la
validación cruzada las transformaciones se ajusten **solo** con los pliegues de
entrenamiento, y permite serializar preparación y modelo como un único objeto para el
despliegue.

In [ ]:
pipeline_preparacion = construir_pipeline_preparacion(
    n_componentes=15,   # la rubrica sugiere explorar entre 10 y 20 para el LSA
    max_features=20000,
    min_df=5,           # descarta terminos que aparecen en menos de 5 documentos
    max_df=0.8,         # descarta terminos presentes en mas del 80% de los documentos
    aplicar_stemming=True,
)
pipeline_preparacion

In [ ]:
%%time
# Se ajusta SOLO con entrenamiento y despues se aplica a prueba
X_train_lsa = pipeline_preparacion.fit_transform(X_train)
X_test_lsa = pipeline_preparacion.transform(X_test)

print(f"Vocabulario TF-IDF: {len(pipeline_preparacion.named_steps['tfidf'].get_feature_names_out()):,} terminos")
print(f"Dimension tras el LSA: {X_train_lsa.shape}")
print(f"Varianza explicada acumulada: {pipeline_preparacion.named_steps['lsa'].explained_variance_ratio_.sum():.1%}")

**✍️ Justificación (completar).** Argumenta las decisiones de esta sección: por qué
TF-IDF frente a conteos simples, qué efecto tuvieron `min_df` y `max_df` sobre el tamaño
del vocabulario, si el stemming ayudó o no, y cómo elegiste el número de componentes.
La rúbrica asigna 30% a estas justificaciones.

## 5. Modelo LSA: extracción e interpretación de tópicos

Cada componente del SVD es una dirección en el espacio de términos. Las palabras con mayor
peso dentro de una componente son las que la definen, y funcionan como un "tópico"
latente. Abajo se listan para inspeccionarlos.

In [ ]:
topicos = terminos_por_topico(pipeline_preparacion, n_terminos=12)
display(topicos.style.format({"varianza_explicada": "{:.2%}"}))

In [ ]:
# Varianza explicada por componente: ayuda a sustentar el numero de componentes elegido
lsa = pipeline_preparacion.named_steps["lsa"]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
ax1.bar(range(1, len(lsa.explained_variance_ratio_) + 1), lsa.explained_variance_ratio_, color="#4C78A8")
ax1.set(xlabel="Componente", ylabel="Varianza explicada", title="Varianza por componente")
ax2.plot(range(1, len(lsa.explained_variance_ratio_) + 1), np.cumsum(lsa.explained_variance_ratio_), marker="o", color="#E45756")
ax2.set(xlabel="Numero de componentes", ylabel="Varianza acumulada", title="Varianza explicada acumulada")
plt.tight_layout()
plt.show()

**✍️ Interpretación cualitativa (completar — 15% de la nota).**

Elige **al menos 5 tópicos** y relaciónalos con los ODS. Una tabla como esta funciona bien:

| Tópico | Términos dominantes | ODS relacionado | Lectura |
|---|---|---|---|
| 1 | *(completar)* | *(completar)* | *(completar)* |
| 2 | | | |
| 3 | | | |
| 4 | | | |
| 5 | | | |

Ten en cuenta dos cosas al interpretar:
- La **primera componente** suele capturar el vocabulario común a todo el corpus y no
  corresponde a ningún ODS en particular; conviene decirlo explícitamente.
- Los términos aparecen **stemmizados** (`educ`, `sosten`), lo cual es esperable. Si
  prefieres tópicos más legibles, puedes reconstruirlos con `aplicar_stemming=False`.

## 6. Modelo de clasificación

Se comparan dos familias de modelos lineales, apropiadas para un espacio denso y de baja
dimensión como el que produce el LSA:

- **Regresión logística multinomial**: expone `predict_proba`, lo que permite mostrar el
  grado de confianza y las alternativas en la aplicación de despliegue.
- **SVM lineal (`LinearSVC`)**: suele rendir bien en clasificación de textos, pero no
  entrega probabilidades de forma directa.

En ambos casos se usa `class_weight="balanced"` para compensar el desbalance detectado
en la sección 2.

In [ ]:
# El pipeline completo anida la preparacion y el clasificador en un solo objeto.
# Asi la busqueda de hiperparametros puede optimizar TAMBIEN los pasos de preparacion
# (por ejemplo el numero de componentes del LSA), sin fugas entre pliegues.
pipeline_completo = Pipeline(steps=[
    ("preparacion", construir_pipeline_preparacion()),
    ("clasificador", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED)),
])

pipeline_completo

In [ ]:
%%time
# Malla deliberadamente pequena: cada combinacion vuelve a limpiar y vectorizar los
# textos, por lo que el costo crece rapido. Amplia la malla cuando ya funcione.
malla = {
    "preparacion__lsa__n_components": [10, 15, 20],
    "clasificador__C": [1.0, 10.0],
}

busqueda = GridSearchCV(
    pipeline_completo,
    param_grid=malla,
    scoring="f1_macro",             # metrica principal: trata igual a todas las clases
    cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED),
    n_jobs=-1,
    verbose=2,
)
busqueda.fit(X_train, y_train)

print(f"\nMejores hiperparametros: {busqueda.best_params_}")
print(f"Mejor F1 macro en validacion cruzada: {busqueda.best_score_:.4f}")

In [ ]:
# Resultados completos de la busqueda, ordenados por desempeno
resultados = pd.DataFrame(busqueda.cv_results_)
columnas = [c for c in resultados.columns if c.startswith("param_")] + ["mean_test_score", "std_test_score"]
display(resultados[columnas].sort_values("mean_test_score", ascending=False).round(4))

**✍️ Justificación (completar — 30% de la nota).** Explica por qué elegiste el algoritmo
final, por qué el **F1 macro** es la métrica adecuada aquí (y no la exactitud), y qué
muestra la tabla anterior sobre el efecto del número de componentes. Si probaste
`LinearSVC` u otro modelo, reporta la comparación.

## 7. Evaluación sobre el conjunto de prueba

El conjunto de prueba no participó en el ajuste de las transformaciones ni en la búsqueda
de hiperparámetros, así que estas cifras estiman el desempeño sobre textos nuevos.

In [ ]:
modelo_final = busqueda.best_estimator_
y_pred = modelo_final.predict(X_test)

print(f"F1 macro:      {f1_score(y_test, y_pred, average='macro'):.4f}")
print(f"F1 ponderado:  {f1_score(y_test, y_pred, average='weighted'):.4f}")
print()
print(classification_report(y_test, y_pred, target_names=etiquetas_ods(sorted(y.unique()))))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 8))
ConfusionMatrixDisplay(
    confusion_matrix(y_test, y_pred, normalize="true"),
    display_labels=sorted(y.unique()),
).plot(ax=ax, cmap="Blues", values_format=".2f", colorbar=False)
ax.set(xlabel="ODS predicho", ylabel="ODS real", title="Matriz de confusion normalizada por fila")
plt.tight_layout()
plt.show()

**✍️ Análisis (completar).** ¿Qué ODS se confunden entre sí y por qué tiene sentido
temáticamente? (El 1 y el 10 comparten vocabulario sobre pobreza y desigualdad; el 13, 14
y 15 comparten el ambiental.) ¿Las clases minoritarias son las de peor desempeño?

### 7.1 Clasificación de textos individuales del conjunto de prueba

La rúbrica exige mostrar explícitamente la clasificación de **al menos cuatro textos**
que no se usaron en el aprendizaje.

In [ ]:
muestra = X_test.sample(4, random_state=SEED)

for posicion, (indice, texto) in enumerate(muestra.items(), start=1):
    real = int(y_test.loc[indice])
    predicho = int(modelo_final.predict([texto])[0])
    acierto = "CORRECTO" if real == predicho else "INCORRECTO"

    print(f"{'=' * 90}")
    print(f"TEXTO {posicion} (indice {indice})")
    print(f"{'=' * 90}")
    print(texto[:500] + ("..." if len(texto) > 500 else ""))
    print(f"\n  ODS real:     {real} - {NOMBRES_ODS[real]}")
    print(f"  ODS predicho: {predicho} - {NOMBRES_ODS[predicho]}   [{acierto}]")

    if hasattr(modelo_final, "predict_proba"):
        probabilidades = modelo_final.predict_proba([texto])[0]
        top3 = np.argsort(probabilidades)[::-1][:3]
        print("  Top 3:", ", ".join(
            f"ODS {modelo_final.classes_[i]} ({probabilidades[i]:.1%})" for i in top3
        ))
    print()

## 8. Serialización del modelo (requisito del bono con Streamlit)

Se guarda **un solo artefacto** con el pipeline completo, preparación incluida. Esa es la
forma de garantizar que el texto que escriba el usuario en la aplicación reciba
exactamente el mismo tratamiento que los textos de entrenamiento.

`LimpiadorTextoEspanol` está definido en `src/DataPreprocessing.py` y no en este notebook
justamente por esto: al deserializar, `joblib` vuelve a importar la clase desde su módulo.
Si estuviera definida en una celda, la aplicación no podría cargar el modelo.

In [ ]:
ruta_modelo = RAIZ / "resources" / "models" / "modelo_ods.joblib"
ruta_modelo.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(modelo_final, ruta_modelo, compress=3)

print(f"Modelo guardado en: {ruta_modelo}")
print(f"Tamano: {ruta_modelo.stat().st_size / 1e6:.1f} MB")

In [ ]:
# Verificacion de ida y vuelta: se recarga el artefacto y se comprueba que predice igual
from src.ModelController import ModelController

controlador = ModelController(ruta_modelo)
prueba = "El acceso al agua potable y al saneamiento sigue siendo limitado en zonas rurales."
print(controlador.predecir(prueba))

## 9. Conclusiones

**✍️ Completar.** Recoge en pocas líneas:

- El desempeño alcanzado y qué métrica lo respalda.
- Qué aportó la reducción de dimensionalidad con LSA, en desempeño y en interpretabilidad.
- Qué tópicos resultaron más claros frente a los ODS y cuáles quedaron ambiguos.
- Limitaciones y qué intentarías después (n-gramas, *embeddings*, agrupar ODS afines).

---

### Antes de entregar

- [ ] Todas las celdas ejecutadas y con su salida visible.
- [ ] Los bloques **✍️** reemplazados por tu argumentación.
- [ ] Exportar a HTML: `jupyter nbconvert --to html notebooks/Microproyecto2.ipynb`
- [ ] Adjuntar `.ipynb` y `.html`.
- [ ] (Bono) URL del despliegue en Streamlit y URL del repositorio de GitHub.